# Run Qwen3.8-27B locally in VS Code Copilot Chat with OpenVINO Model Server (OVMS)

This notebook walks you **step by step** through running the INT4 OpenVINO model
[`Morteza89/qwen3.8-27b-int4-ov`](https://huggingface.co/Morteza89/qwen3.8-27b-int4-ov) **on your own Intel AI PC** and using it
as a model in the **VS Code Copilot Chat model picker**, next to (or instead of) the cloud models.

### What you will build

```
 VS Code Copilot Chat  ──(OpenAI Chat Completions API)──►  OVMS on 127.0.0.1:8000  ──►  OpenVINO  ──►  Intel Arc iGPU
 (model picker: "Qwen3.8 27B (OVMS iGPU)")                  (openvino_model_server)                    (Qwen3.8-27B INT4)
```

* **OpenVINO GenAI**: used once to check that the model runs on your GPU.
* **OpenVINO Model Server (OVMS)**: serves the model with an **OpenAI-compatible REST API** (`/v3/chat/completions`), including **tool calling** (needed for Copilot *Agent* mode) and **reasoning** parsing.
* **VS Code Copilot "Custom Endpoint" provider**: adds your local server to the model picker.

### Why do this?
* **No per-token cost**: everything runs on your PC.
* **Privacy**: prompts and code stay on your PC.
* **Works offline** once the model is downloaded.

> Every step below was tested end-to-end on Windows 11 with an Intel Core Ultra X7 358H / Intel Arc B390 iGPU and 64 GB RAM.
> The problems we hit along the way and how we fixed them are listed in the **Troubleshooting** section at the end.

## 0. Requirements

| Item | Requirement / tested value |
|---|---|
| OS | Windows 11 |
| CPU / GPU | Intel Core Ultra with Intel Arc iGPU (tested: Core Ultra X7 358H + **Arc B390**). A discrete Intel Arc GPU also works. |
| RAM | **32 GB minimum, 64 GB recommended.** The model uses about **18 GB** of RAM (an iGPU shares system memory). |
| Disk | About **25 GB** free (model plus OVMS plus compile cache) |
| GPU driver | Latest Intel Arc / Graphics driver (tested: 32.0.101.8356 and 32.0.101.8860) |
| Python | **3.12** (64-bit) |
| VS Code | Recent version with **GitHub Copilot Chat** signed in, plus the **Jupyter** and **Python** extensions to run this notebook |
| Copilot plan | You must be signed in to Copilot. On **Business/Enterprise** plans, your org admin must allow "bring your own model/key" (see step 9.1 to check). |

> **NPU is not supported for this model.** The model uses *asymmetric* INT4 weights, and the OpenVINO NPU path currently needs symmetric INT4. Use **GPU** (fastest) or CPU.

## 1. One-time bootstrap: create a Python virtual environment for this notebook

This notebook needs a Python kernel, so create the environment **once, from a PowerShell terminal**, before running any cell.

```powershell
# 1) Create a working folder and a virtual environment (Python 3.12)
New-Item -ItemType Directory -Force C:\ov_local_llm | Out-Null
py -3.12 -m venv C:\ov_local_llm\.venv
#   (if the "py" launcher is missing, use the full path to python.exe 3.12 instead)

# 2) Install the Jupyter kernel package into it
C:\ov_local_llm\.venv\Scripts\python.exe -m pip install --upgrade pip ipykernel
```

Then in VS Code:
1. Open this notebook.
2. Click **Select Kernel** (top-right) → **Python Environments** → choose `C:\ov_local_llm\.venv\Scripts\python.exe`.
3. Run the cells **from top to bottom**.

> Why a separate venv? The model was converted with a specific OpenVINO version (**2026.4.0rc1**). Keeping it isolated avoids conflicts with other Python projects.

## 2. Configuration

All paths and names are set in **one cell**. Change them here if you want different folders; every later cell uses these variables.

> **Always run this cell first**, including after a kernel restart or when you come back later and jump straight to a section (for example "Start the server" or "Stop the server"). Otherwise later cells fail with `NameError: name 'BASE_DIR' is not defined`.

* `MODEL_DIR`: where the model is downloaded. **If you already have the model on disk, point this to that folder** and the download step will skip files that already exist.
* `OVMS_DIR`: where OVMS is extracted.
* `SERVED_MODEL_NAME`: the model **ID** that OVMS exposes. VS Code must use **exactly** this string.
* `HOST = "127.0.0.1"`: the server listens **only on your PC**. Don't change this to `0.0.0.0` unless you know why, because the server has no API key by default.

In [ ]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

HF_REPO_ID = "Morteza89/qwen3.8-27b-int4-ov"

BASE_DIR = Path(r"C:\ov_local_llm")
MODEL_DIR = BASE_DIR / "models" / "qwen3.8-27b-int4-ov"
OV_CACHE_DIR = BASE_DIR / "ov_cache"          # GPU compile cache -> much faster restarts

OVMS_VERSION = "2026.4.0"                     # first OVMS release with Qwen3.8-27B support
OVMS_DIR = BASE_DIR / "ovms"                  # the OVMS zip contains a top-level "ovms" folder

SERVED_MODEL_NAME = "qwen3.8-27b-ov"          # model id used by VS Code
HOST = "127.0.0.1"
PORT = 8000
TARGET_DEVICE = "GPU"                         # "GPU" (iGPU/dGPU) or "CPU"; NPU is not supported for this model

API_BASE = f"http://{HOST}:{PORT}"
CHAT_URL = f"{API_BASE}/v3/chat/completions"

for d in (BASE_DIR, MODEL_DIR.parent, OV_CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Model dir :", MODEL_DIR)
print("OVMS dir  :", OVMS_DIR)
print("Endpoint  :", CHAT_URL)

## 3. Install OpenVINO packages

We install the **same OpenVINO versions the model was converted with** (from `conversion_metadata.json` in the model repo):

| Package | Version |
|---|---|
| `openvino` | 2026.4.0rc1 |
| `openvino-genai` | 2026.4.0.0rc1 |
| `openvino-tokenizers` | 2026.4.0.0rc1 |

These are pre-release builds, so we add the **OpenVINO nightly index** and `--pre`.
We don't need `torch` or `optimum` here, because the model is **already converted** and only needs the runtime.

> These packages are only needed for the local check in **step 5**. OVMS (step 6) ships its own OpenVINO runtime.

In [ ]:
%pip install --pre --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly openvino==2026.4.0rc1 openvino-tokenizers==2026.4.0.0rc1 openvino-genai==2026.4.0.0rc1 huggingface_hub numpy pillow requests psutil

## 4. Check your hardware

This cell checks that:
* OpenVINO can see your **GPU** (the list should show `GPU` with a name like *Intel(R) Arc(TM) B390 GPU*),
* you have enough **RAM** and **disk space**,
* which **GPU driver** is installed.

If `GPU` is missing, install or update the Intel graphics driver and **reboot**.
You may also see `NPU` in the list. That's normal, but this model can't run on it.

In [ ]:
import shutil
import subprocess
import openvino as ov
import psutil

core = ov.Core()
print("OpenVINO:", ov.__version__)
for dev in core.available_devices:
    print(f"  {dev:5s} -> {core.get_property(dev, 'FULL_DEVICE_NAME')}")

ram_gb = psutil.virtual_memory().total / 2**30
free_gb = shutil.disk_usage(BASE_DIR).free / 2**30
print(f"\nRAM total : {ram_gb:.1f} GB  (need >= 32, 64 recommended)")
print(f"Disk free : {free_gb:.1f} GB  (need ~25)")

print("\nGPU driver(s):")
print(subprocess.run(
    ["powershell", "-NoProfile", "-Command",
     "Get-CimInstance Win32_VideoController | Select-Object Name, DriverVersion | Format-Table -AutoSize"],
    capture_output=True, text=True).stdout)

assert TARGET_DEVICE == "CPU" or TARGET_DEVICE in core.available_devices, \
    f"{TARGET_DEVICE} not visible to OpenVINO - update the Intel GPU driver and reboot."

## 5. Download the model from Hugging Face

`snapshot_download` downloads every file in the repo into `MODEL_DIR` (about 15–17 GB). If it's interrupted, **run it again**: it resumes and skips files that are already complete.

What's inside (OpenVINO IR format, `.xml` graph plus `.bin` weights):
* `openvino_language_model.*`: the 27B language model (INT4)
* `openvino_vision_embeddings*`: the vision encoder (this is a **vision-language model**, so you can also paste images in chat)
* `openvino_tokenizer.*` / `openvino_detokenizer.*`: tokenizers converted to OpenVINO. **OVMS needs these.**
* `chat_template.jinja`: the chat template, which OVMS uses to format messages and tool calls

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id=HF_REPO_ID, local_dir=MODEL_DIR)

required = ["openvino_language_model.xml", "openvino_language_model.bin",
            "openvino_tokenizer.xml", "openvino_detokenizer.xml",
            "openvino_vision_embeddings_model.xml", "chat_template.jinja", "config.json"]
missing = [f for f in required if not (MODEL_DIR / f).exists()]
size_gb = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file()) / 2**30
print(f"Model folder size: {size_gb:.1f} GB")
print("Missing files:", missing or "none - OK")

## 6. Check that the model runs on the GPU (OpenVINO GenAI)

Before adding a server, confirm that the model **loads and generates on your GPU** with OpenVINO GenAI's `VLMPipeline`.
If this step fails, OVMS will fail too, and this step's error messages are easier to read.

* Loading reads about 17 GB from disk and compiles the model for the GPU, so expect this to take from tens of seconds to a few minutes, depending on your disk and GPU.
* We send one image plus a prompt (the same test as the model card) and print speed metrics:
  * **TTFT** (time to first token): how long until the answer starts
  * **TPOT** (time per output token) and **throughput** (tokens/s)
* Reference on Arc B390: about **7.4 tokens/s**, and a TTFT of about 6.3 s for a prompt of about 2.2k tokens.

At the end we **delete the pipeline to free about 18 GB of RAM**, because OVMS loads its own copy next.

In [ ]:
import gc
import time
from io import BytesIO

import numpy as np
import openvino_genai as ov_genai
import requests
from PIL import Image

IMAGE_URL = ("https://huggingface.co/datasets/huggingface/documentation-images/"
             "resolve/main/transformers/tasks/ai2d-demo.jpg")
image = Image.open(BytesIO(requests.get(IMAGE_URL, timeout=30).content)).convert("RGB")
image_tensor = ov.Tensor(np.asarray(image)[None])

t0 = time.perf_counter()
pipe = ov_genai.VLMPipeline(MODEL_DIR, TARGET_DEVICE)
print(f"Pipeline loaded on {TARGET_DEVICE} in {time.perf_counter() - t0:.1f} s\n")

def streamer(subword: str) -> bool:
    print(subword, end="", flush=True)
    return False  # False = keep generating

result = pipe.generate("Describe this image in one paragraph.",
                       image=image_tensor, max_new_tokens=64, streamer=streamer)

pm = result.perf_metrics
print("\n\n--- performance ---")
print(f"Input tokens : {pm.get_num_input_tokens()}")
print(f"TTFT         : {pm.get_ttft().mean:.0f} ms")
print(f"TPOT         : {pm.get_tpot().mean:.0f} ms/token")
print(f"Throughput   : {pm.get_throughput().mean:.2f} tokens/s")

del pipe, result
gc.collect()
print("\nPipeline released - RAM freed for OVMS.")

## 7. Download and install OpenVINO Model Server (Windows)

VS Code can't talk to Python directly. It needs an **HTTP server with an OpenAI-compatible API**. OVMS provides that.

Important details:
* **Use OVMS 2026.4.0 or newer.** It's the first release that officially supports **Qwen3.8-27B**, and its OpenVINO runtime matches the model's conversion version. Older OVMS versions fail to load the model.
* Use the **`python_on`** package. It renders chat templates with full **Jinja** support, which Qwen's template (with tool calls) needs.
* We **verify the SHA-256 checksum** before extracting. Always do this for downloaded executables.
* The package is extracted to `OVMS_DIR`. No admin rights and no system-wide install are needed.

In [ ]:
import hashlib
import urllib.request
import zipfile

pkg = f"ovms_windows_{OVMS_VERSION}_python_on.zip"
url = f"https://storage.openvinotoolkit.org/repositories/openvino_model_server/packages/{OVMS_VERSION}/{pkg}"
zip_path = BASE_DIR / pkg

if not zip_path.exists():
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path)

expected = urllib.request.urlopen(url + ".sha256").read().decode().split()[0].lower()
sha = hashlib.sha256()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        sha.update(chunk)
actual = sha.hexdigest().lower()
print("expected:", expected)
print("actual  :", actual)
if actual != expected:
    zip_path.unlink()
    raise RuntimeError("SHA-256 mismatch - corrupted download deleted, run this cell again.")

with zipfile.ZipFile(zip_path) as z:
    z.extractall(BASE_DIR)
assert (OVMS_DIR / "ovms.exe").exists(), f"ovms.exe not found in {OVMS_DIR}"
print("\nOVMS extracted to", OVMS_DIR)

### 7.1 Verify the OVMS installation

OVMS must be started **after running its `setupvars` script**, which puts OVMS and its bundled Python on `PATH`.

> **Pitfall we hit:** `setupvars` sets `PYTHONHOME` to OVMS's *embedded* Python. If you run `setupvars.ps1` in a terminal and then try to use your **venv** Python in the *same* terminal, it fails with
> `ModuleNotFoundError: No module named 'encodings'`.
> **Fix:** always run OVMS in its **own** terminal or window (as this notebook does), and your venv in another.

The cell below runs `ovms --version` in a separate `cmd` process, so your notebook environment isn't affected.

In [ ]:
clean_env = {k: v for k, v in os.environ.items() if k not in ("PYTHONHOME", "PYTHONPATH", "VIRTUAL_ENV")}
out = subprocess.run(
    ["cmd", "/c", f'call "{OVMS_DIR}\\setupvars.bat" >nul && "{OVMS_DIR}\\ovms.exe" --version'],
    capture_output=True, text=True, env=clean_env)
print(out.stdout or out.stderr)

## 8. Start the model server

We write a small **`start_ovms.bat`** launcher. It's reusable: **each day, just double-click it** before opening VS Code.

What each flag does:

| Flag | Why |
|---|---|
| `--rest_port 8000` | HTTP port for the OpenAI-compatible API |
| `--rest_bind_address 127.0.0.1` | **Security:** listen only on this PC. Without it OVMS binds to `0.0.0.0`, so anyone on your network could use it (no API key is set). |
| `--model_name qwen3.8-27b-ov` | The model ID that clients (VS Code) must send |
| `--model_path <folder>` | Serve the local model folder directly, with no config file needed |
| `--task text_generation` | Enables `/v3/chat/completions`. OVMS detects that this is a **vision-language model** and uses the VLM continuous-batching pipeline. |
| `--target_device GPU` | Run on the Intel GPU (use `CPU` if needed) |
| `--enable_prefix_caching true` | Reuses already-processed prompt prefixes. Copilot resends a **large system prompt** every turn, so this speeds up follow-up turns a lot. |
| `--cache_dir <folder>` | Stores the compiled GPU model so later starts skip GPU compilation |

**Tool calling and reasoning:** we don't pass `--tool_parser` / `--reasoning_parser`. OVMS 2026.4 **detects them automatically** for this model. The OVMS log shows:
```
Auto-detected tool_parser: qwen3coder
Auto-detected reasoning_parser: qwen3
```
We also verify this in step 8.3: tool calls come back as proper `tool_calls` and thinking goes to `reasoning_content`.

The next cell writes the `.bat` file and **opens OVMS in a new console window**. **Keep that window open.** Closing it stops the model, and it disappears from VS Code.

In [ ]:
bat_path = BASE_DIR / "start_ovms.bat"
bat_path.write_text(f"""@echo off
title OVMS - {SERVED_MODEL_NAME}
set PYTHONHOME=
set PYTHONPATH=
call "{OVMS_DIR}\\setupvars.bat"
"{OVMS_DIR}\\ovms.exe" ^
  --rest_port {PORT} ^
  --rest_bind_address {HOST} ^
  --model_name {SERVED_MODEL_NAME} ^
  --model_path "{MODEL_DIR}" ^
  --task text_generation ^
  --target_device {TARGET_DEVICE} ^
  --enable_prefix_caching true ^
  --cache_dir "{OV_CACHE_DIR}" ^
  --log_level INFO
pause
""", encoding="ascii")
print(bat_path.read_text())

os.startfile(bat_path)  # opens a new console window running OVMS
print(f"OVMS launched in a new window. Next time just double-click: {bat_path}")

> **Note:** if the OVMS window doesn't open or is closed, start it manually by double-clicking `C:\ov_local_llm\start_ovms.bat`.

### 8.1 Wait until the model is loaded

The REST port opens within seconds, but the model needs extra time to load. In the OVMS window you'll see
`Initializing Visual Language Model Continuous Batching servable` for a while. **That's normal, not a hang.**
On the reference Arc B390 system, the first start took about **30 seconds**. Slower disks or GPUs can take longer.

The model is ready when the log shows:
```
Mediapipe: qwen3.8-27b-ov state changed to: AVAILABLE
```

This cell polls the readiness endpoint `/v2/health/ready` until the model is fully loaded (up to 30 minutes).

In [ ]:
import time
import requests

t0 = time.time()
while True:
    try:
        if requests.get(f"{API_BASE}/v2/health/ready", timeout=5).status_code == 200:
            break
    except requests.ConnectionError:
        pass
    elapsed = time.time() - t0
    if elapsed > 30 * 60:
        raise TimeoutError("OVMS not ready after 30 min - check the OVMS window for errors.")
    print(f"\rwaiting for model to load... {elapsed:5.0f} s", end="")
    time.sleep(10)

print(f"\nOVMS ready after {time.time() - t0:.0f} s")
print(requests.get(f"{API_BASE}/v3/models", timeout=10).json())

### 8.2 Test a chat request

This sends a normal OpenAI-style chat request. You should get Python code back, with `finish_reason: stop`.
Because this is a **thinking** model, OVMS returns the model's reasoning separately in `reasoning_content` and the answer in `content`.

> **Pitfall we hit (Windows PowerShell 5.1):** testing with `curl.exe ... -d '{\"model\": ...}'` fails with
> `The file is not valid json` and `curl: (6) Could not resolve host: ...`, because PowerShell 5.1 breaks the quotes inside JSON arguments.
> Use Python (as here) or PowerShell's `Invoke-RestMethod`:
> ```powershell
> $body = @{ model='qwen3.8-27b-ov'; max_tokens=64; messages=@(@{role='user'; content='Write a Python hello world'}) } | ConvertTo-Json -Depth 10
> Invoke-RestMethod http://127.0.0.1:8000/v3/chat/completions -Method Post -ContentType 'application/json' -Body $body
> ```

In [ ]:
import json

payload = {
    "model": SERVED_MODEL_NAME,
    "max_tokens": 512,
    "messages": [{"role": "user", "content": "Write a Python hello world"}],
}
t0 = time.perf_counter()
r = requests.post(CHAT_URL, json=payload, timeout=600)
r.raise_for_status()
resp = r.json()
elapsed = time.perf_counter() - t0

msg = resp["choices"][0]["message"]
print("finish_reason    :", resp["choices"][0]["finish_reason"])
print("reasoning (short):", (msg.get("reasoning_content") or "")[:300], "...")
print("\nanswer:\n", msg.get("content"))
usage = resp.get("usage", {})
print(f"\nusage: {usage}   elapsed: {elapsed:.1f} s   "
      f"~{usage.get('completion_tokens', 0) / elapsed:.1f} tokens/s (incl. prompt processing)")

### 8.3 Test tool calling (required for Copilot Agent mode)

Copilot **Agent** mode works by asking the model to call tools (read file, edit file, run terminal, ...).
This only works if the server returns **structured `tool_calls`** rather than plain text.

Expected result: `finish_reason: tool_calls` and a call like `get_weather({"city": "Paris"})`.
If you get plain text instead, the tool parser wasn't detected. Add `--tool_parser qwen3coder --reasoning_parser qwen3` to `start_ovms.bat` and restart it.

In [ ]:
payload = {
    "model": SERVED_MODEL_NAME,
    "max_tokens": 512,
    "messages": [{"role": "user", "content": "What is the weather in Paris? Use the tool."}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {"type": "object",
                           "properties": {"city": {"type": "string"}},
                           "required": ["city"]},
        },
    }],
}
resp = requests.post(CHAT_URL, json=payload, timeout=600).json()
choice = resp["choices"][0]
print("finish_reason:", choice["finish_reason"])
for call in choice["message"].get("tool_calls") or []:
    print("tool call    :", call["function"]["name"], call["function"]["arguments"])
assert choice["finish_reason"] == "tool_calls", "No structured tool call - see the note above."
print("\nTool calling works - the model can be used in Copilot Agent mode.")

## 9. Add the model to VS Code Copilot Chat

### 9.1 (Optional) Check that your Copilot policy allows custom models

If your Copilot license comes from a company (Business/Enterprise), an admin can disable "bring your own key" (BYOK) models.
This cell searches the latest **GitHub Copilot Chat** log for the list of BYOK providers.

* If you see `customendpoint` in `BYOK: registered ... provider(s)`, **you're good**.
* `Copilot preview features are disabled by organizational policy` is only a warning. It hides *preview* providers (for example the deprecated "OpenAI Compatible") but **not** "Custom Endpoint".

To view the log manually: **Ctrl+Shift+P** → `Developer: Show Logs...` → **GitHub Copilot Chat**, then press **Ctrl+F** and search for `BYOK`.

In [ ]:
import re

logs_root = Path(os.environ["APPDATA"]) / "Code" / "logs"
latest = max(logs_root.iterdir(), key=lambda p: p.stat().st_mtime)
hits = set()
for log in latest.rglob("GitHub Copilot Chat.log"):
    for line in log.read_text(encoding="utf-8", errors="ignore").splitlines():
        if re.search(r"BYOK: registered|BYOK models available|organizational policy", line):
            hits.add(re.sub(r"^\S+ \S+ ", "", line))
print("\n".join(sorted(hits)) or "No Copilot Chat log found - open Copilot Chat in VS Code once and rerun.")

### 9.2 Create the "Custom Endpoint" group (in the VS Code UI)

> **Pitfall we hit #1:** older guides tell you to add `github.copilot.chat.customOAIModels` to `settings.json`. **Current VS Code ignores that setting.** Custom models are now managed in the **Language Models** editor and stored in `chatLanguageModels.json`.
>
> **Pitfall we hit #2:** "OpenAI Compatible" is **deprecated** and may be hidden. Use **Custom Endpoint**.

Do this **once**, by hand, because the API key goes into VS Code's secure secret storage:

1. Open the Chat view → click the **model picker** → **Other Models** → **gear icon** (Manage Models).
   *(or Ctrl+Shift+P → `Chat: Manage Language Models`)*
2. Click **Add Models** → choose **Custom Endpoint**.
3. **Group Name** → type `OVMS Local` → Enter.
   > **Pitfall we hit #3:** this prompt asks only for a **label**. Don't paste the URL here. Also make sure the **search box** at the top of the Language Models editor is **empty**, otherwise the table looks empty.
4. **API Key** → type any non-empty text, for example `ovms-local` → Enter.
   (The field can't be empty. OVMS ignores the key because we didn't configure one.)
5. **API Type**, if asked → **Chat Completions**.

VS Code now opens `chatLanguageModels.json` with a new `OVMS Local` entry that has **no models yet**. Continue with 9.3.

### 9.3 Add the model to the group (automated)

This cell opens `%APPDATA%\Code\User\chatLanguageModels.json`, finds the `OVMS Local` group, and adds the model definition.
It **makes a backup first** (`chatLanguageModels.json.bak`).

Field meanings:

| Field | Value | Why |
|---|---|---|
| `id` | `qwen3.8-27b-ov` | **Must match** `--model_name` exactly, or you get "model not found" |
| `url` | `http://127.0.0.1:8000/v3/chat/completions` | Full Chat Completions URL. We use `127.0.0.1` rather than `localhost` because the server listens only on IPv4 and `localhost` may resolve to IPv6 `::1`. |
| `toolCalling` | `true` | Enables **Agent** mode (tested in 8.3) |
| `vision` | `true` | It's a VLM, so you can attach images or screenshots in chat |
| `thinking` | `true` | Shows the model's reasoning in a collapsible "thinking" section |
| `maxInputTokens` / `maxOutputTokens` | 32768 / 4096 | Keeps memory use reasonable on an iGPU (the KV cache uses shared RAM) |
| `modelOptions.temperature` / `top_p` | 0.6 / 0.95 | Qwen's recommended thinking-mode sampling. The model's default `temperature` is **1.0**, which made it **reply in Chinese** at random (see step 10). |

> If you use **VS Code Insiders**, change `"Code"` to `"Code - Insiders"` in the path below.
> If you use a non-default VS Code *profile*, the file is under `...\Code\User\profiles\<id>\`.

In [ ]:
GROUP_NAME = "OVMS Local"
lm_file = Path(os.environ["APPDATA"]) / "Code" / "User" / "chatLanguageModels.json"

model_entry = {
    "id": SERVED_MODEL_NAME,
    "name": "Qwen3.8 27B (OVMS iGPU)",
    "url": CHAT_URL,
    "apiType": "chat-completions",
    "toolCalling": True,
    "vision": True,
    "thinking": True,
    "maxInputTokens": 32768,
    "maxOutputTokens": 4096,
    "modelOptions": {"temperature": 0.6, "top_p": 0.95},
}

groups = json.loads(lm_file.read_text(encoding="utf-8")) if lm_file.exists() else []
group = next((g for g in groups if g.get("vendor") == "customendpoint" and g.get("name") == GROUP_NAME), None)

if group is None:
    print(f"Group '{GROUP_NAME}' not found in {lm_file}.")
    print("Do step 9.2 first (Add Models -> Custom Endpoint), then rerun this cell.")
    print("\nOr paste this into the group's \"models\" array manually:\n")
    print(json.dumps([model_entry], indent=2))
else:
    shutil.copy2(lm_file, lm_file.with_suffix(".json.bak"))
    models = [m for m in group.get("models", []) if m.get("id") != SERVED_MODEL_NAME]
    group["models"] = models + [model_entry]
    lm_file.write_text(json.dumps(groups, indent="\t"), encoding="utf-8")
    print(f"Added '{model_entry['name']}' to group '{GROUP_NAME}' in {lm_file}")
    print(json.dumps(group | {"apiKey": "<stored in VS Code secret storage>"}, indent=2))

### 9.4 Select the model in Chat

1. **Ctrl+Shift+P** → `Developer: Reload Window`.
2. In the **Language Models** editor, make sure the model under **OVMS Local** is **visible** (eye icon on).
3. Chat view → **model picker** → **Other Models** → **Qwen3.8 27B (OVMS iGPU)**.
4. Start a **new** chat and ask something like: *"Do you run locally? Which model are you?"*

Tips for daily use:
* **Ask mode** is noticeably faster than **Agent** mode, because Agent sends many tool definitions, which means a much longer prompt to process.
* The **first message of a session is the slowest** (in our test, about 1m 20s on the iGPU, mostly processing Copilot's large system prompt). Follow-ups are faster thanks to prefix caching.
* Keep the **OVMS window open** while you work.

## 10. Make the model always answer in English

**Problem we hit:** the first reply in VS Code came back in **Chinese**, even though the question was in English.
Qwen models were trained on a lot of Chinese text, and several things pushed it that way:
* the model's default `temperature` is **1.0**, which is quite random, and INT4 quantization adds a little more noise,
* Copilot's long system prompt doesn't say which language to use.

**Fixes (use all three):**
1. **Lower the temperature**: already done in step 9.3 (`temperature 0.6`, `top_p 0.95`).
2. **Set Copilot's response language.** Open **user** `settings.json` (Ctrl+Shift+P → `Preferences: Open User Settings (JSON)`) and add:
   ```json
   "github.copilot.chat.localeOverride": "en"
   ```
3. **Add a user-level instruction** that is included with every chat request. The cell below creates it at `%APPDATA%\Code\User\prompts\english.instructions.md`.

Then **reload the window** and start a **new** chat session. An old session that already contains Chinese text can pull the model back into Chinese.
If it still drifts sometimes, lower `temperature` to `0.3` in `chatLanguageModels.json`.

In [ ]:
prompts_dir = Path(os.environ["APPDATA"]) / "Code" / "User" / "prompts"
prompts_dir.mkdir(parents=True, exist_ok=True)
instr = prompts_dir / "english.instructions.md"
if instr.exists():
    print("Already exists:", instr)
else:
    instr.write_text(
        '---\napplyTo: "**"\n---\n'
        "Always respond in English, including explanations and code comments, "
        "regardless of the model's default language.\n",
        encoding="utf-8")
    print("Created:", instr)
print(instr.read_text(encoding="utf-8"))

## 11. Daily usage

**Every day:**
1. Double-click `C:\ov_local_llm\start_ovms.bat` and wait until the log settles. It's much faster than the first time thanks to `ov_cache`.
2. Open VS Code → Chat → pick **Qwen3.8 27B (OVMS iGPU)**.
3. Stop it when you want to free up the RAM and no need of model usage anymore:
```powershell
Get-Process ovms -ErrorAction SilentlyContinue | Stop-Process

**To change settings** (device, context, parsers): stop the server (section 12), edit `start_ovms.bat`, and start it again.

## 12. Stop the server

**Why stop it?** While OVMS is running, the model uses about **18 GB of RAM** and keeps the GPU busy. Stop it when you're done coding, or before gaming, video editing, or running another large model.

**What happens in VS Code:** the model stays in the picker, but requests fail with a *connection* error. Switch the picker back to a cloud model, or start the server again.

Stopping is **safe at any time**. OVMS only *reads* the model files, so nothing gets corrupted. The GPU compile cache in `ov_cache` is kept, so the next start is still fast.

### Option A: from the OVMS window (recommended)
1. Click the console window titled **`OVMS - qwen3.8-27b-ov`**.
2. Press **Ctrl+C**. OVMS shuts down cleanly (you'll see shutdown messages in the log).
3. The window shows `Press any key to continue . . .`. Press any key to close it.

Simply **closing the window (X)** also stops the server.

### Option B: from PowerShell
```powershell
Get-Process ovms -ErrorAction SilentlyContinue | Stop-Process
Get-NetTCPConnection -LocalPort 8000 -State Listen -ErrorAction SilentlyContinue   # no output = stopped
```

### Option C: from this notebook
The next cell finds every running `ovms.exe`, shows its PID, memory use and command line, stops it, and then checks that the port is free and the RAM has been released.

> This stops **all** OVMS processes on your PC. If you run other OVMS servers, stop this one from its window instead (option A).

In [ ]:
import time
import psutil
import requests

procs = [p for p in psutil.process_iter(["pid", "name", "memory_info", "cmdline"])
         if (p.info["name"] or "").lower() == "ovms.exe"]

if not procs:
    print("No ovms.exe process is running - nothing to stop.")
else:
    for p in procs:
        print(f"PID {p.pid}  RAM {p.info['memory_info'].rss / 2**30:.1f} GB")
        print("   ", " ".join(p.info["cmdline"] or [])[:200])
        p.terminate()
    gone, alive = psutil.wait_procs(procs, timeout=30)
    for p in alive:
        p.kill()
    print(f"\nStopped {len(procs)} OVMS process(es).")

time.sleep(2)
try:
    requests.get(f"{API_BASE}/v3/models", timeout=3)
    print(f"WARNING: something is still answering on port {PORT}.")
except requests.ConnectionError:
    print(f"Port {PORT} is free - server is stopped.")
print(f"Available RAM now: {psutil.virtual_memory().available / 2**30:.1f} GB")

### Start again later
Double-click `C:\ov_local_llm\start_ovms.bat`, or rerun the cell in section 8 and then the readiness check in 8.1. Your VS Code setup doesn't need to change.

### Remove everything (optional)
1. Stop the server (above).
2. In VS Code: **Manage Models** → right-click the **OVMS Local** group → remove it.
3. Delete the folder `C:\ov_local_llm` (venv, OVMS, model, cache). This frees about 20 GB.
4. Optional: delete `%APPDATA%\Code\User\prompts\english.instructions.md` and the `github.copilot.chat.localeOverride` setting.

## 13. Troubleshooting (everything we ran into)

| Symptom | Cause | Fix |
|---|---|---|
| Model not in the picker after adding `github.copilot.chat.customOAIModels` to `settings.json` | That setting is ignored by current VS Code | Use **Manage Models → Add Models → Custom Endpoint** (step 9.2). Models live in `chatLanguageModels.json`. |
| No "OpenAI Compatible" in the **Add Models** list | It's deprecated or hidden (preview features can be disabled by org policy) | Use **Custom Endpoint** |
| Language Models editor shows an empty table | Text (for example the URL) left in the **search box** | Clear the search box |
| Group created with the URL as its name | The **Group Name** prompt is only a label | Delete the group, re-add it with a name like `OVMS Local`, and put the URL in `chatLanguageModels.json` |
| API key prompt won't accept an empty value | The field is required | Enter any text (`ovms-local`). OVMS has no key configured. |
| `ModuleNotFoundError: No module named 'encodings'` | `setupvars` set `PYTHONHOME` to OVMS's embedded Python in the same terminal as your venv | Run OVMS in its own window (`start_ovms.bat`) and use a different terminal for the venv |
| `curl: (6) Could not resolve host` / `not valid json` | PowerShell 5.1 breaks JSON quotes passed to `curl.exe` | Use `Invoke-RestMethod` or Python `requests` (step 8.2) |
| OVMS sits at `Initializing Visual Language Model Continuous Batching servable` | The model is still loading and compiling for the GPU (about 30 s on Arc B390) | Wait for `state changed to: AVAILABLE`. If there's no progress after 10+ minutes, check RAM and the GPU driver. |
| Model fails to load in OVMS | OVMS older than 2026.4.0 (mismatched OpenVINO version) | Use OVMS **2026.4.0+** |
| Replies in **Chinese** | Qwen language drift, default temperature 1.0 | Step 10: temperature 0.6, `localeOverride: en`, English instruction file, new chat |
| "Model not found" error in chat | `id` in `chatLanguageModels.json` differs from `--model_name` | Make them identical (`qwen3.8-27b-ov`); check `GET /v3/models` |
| Connection refused from VS Code | OVMS stopped or its window closed, or `localhost` resolved to IPv6 | Start `start_ovms.bat`; use `http://127.0.0.1:8000/...` in the URL |
| Agent mode doesn't edit files / prints JSON as text | Tool parser not active (the log should show `Auto-detected tool_parser: qwen3coder`) | Check step 8.3; add `--tool_parser qwen3coder --reasoning_parser qwen3` |
| NPU errors | Asymmetric INT4 is not supported on NPU | Use `GPU` or `CPU` |
| Very slow first reply in Agent mode | Copilot sends a very large prompt, and the iGPU prefill is slow | Use Ask mode for quick questions, keep prefix caching on, and reuse the same chat session |
| Port 8000 already in use | Another app is using it, or an old OVMS is still running | Stop old OVMS (section 12), or change `PORT` in step 2 and rerun steps 8 and 9.3 |

## 14. Security notes
* The server binds to **127.0.0.1** only. If you ever expose it on the network, add `--api_key_file <file>` and use that key in VS Code.
* The OVMS download is **SHA-256 verified** (step 7).
* This is an **unofficial community conversion** of `Qwen/Qwen3.8-27B` (Apache-2.0). Check generated code as you would with any AI assistant.

**Enjoy your free, private, local coding assistant!**